In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

df = pd.read_csv("../Buoi3/du-lieu/training_set.csv")
print(df.shape)
df.head()

df['van_ban_goc'] = df['van_ban_goc'].fillna("")

X = df["van_ban_goc"]
y = df["nhan"]

print(df["van_ban_goc"].isna().sum())

def mo_hinh():
    return make_pipeline(
        TfidfVectorizer(min_df=2),
        LogisticRegression(max_iter=1000)
    )

f1 = lambda a, b: f1_score(a, b, average="macro", labels=[0, 1], zero_division=0)

(1024, 3)
0


In [ ]:
# 1 va 2. Cắt training_set.csv thành hai phần: học và kiểm định.
Xh, Xk, yh, yk = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
m = mo_hinh()
m.fit(Xh, yh)

pk = m.predict_proba(Xk)[:, 1]
tau = max(
    np.arange(0.05, 0.96, 0.01),
    key=lambda t: f1(yk, (pk >= t).astype(int))
)
print(f"Ngưỡng tối ưu tau = {tau:.2f}")

Ngưỡng tối ưu tau = 0.14


In [22]:
# 3. Huấn luyện lại trên toàn bộ dữ liệu huấn luyện.
m_cuoi = mo_hinh()
m_cuoi.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None


In [ ]:
# 4. Áp τ đã chốt lên tập công khai. Không quét lại.
de = pd.read_csv("../Buoi3/du-lieu/public_test.csv")

p = m_cuoi.predict_proba(de["van_ban_goc"])[:, 1]
nhan = (p >= tau).astype(int)